# Generate Oblique Rotated Test Cases

This notebook creates reproducible oblique variants of the listed NIfTI test images by applying a deterministic random 3D rotation to each image affine around the physical image center. The voxel array is preserved; only the affine is updated so the saved output becomes an oblique test case.


In [1]:
from __future__ import annotations

from pathlib import Path

import nibabel as nib
import numpy as np

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "mni_icbm152_2009c_t1_1mm_img.nii.gz").exists():
    NOTEBOOK_DIR = Path("/homebase/code_sync/myMedIADIRLab/data_io/unittest_image")

INPUT_FILENAMES: tuple[str, ...] = (
    "mni_icbm152_2009c_t1_1mm_img.nii.gz",
    "HN_RRT_Jess_tp1_ct.nii.gz",
    "OASIS_ORG_0001_orig.nii.gz",
)
RNG_SEED: int = 20260401
ANGLE_RANGE_DEGREES: tuple[float, float] = (-22.0, 22.0)


def make_rotation_matrix_xyz(angles_degrees: np.ndarray) -> np.ndarray:
    """Build a 3x3 rotation matrix from XYZ Euler angles in degrees.

    Parameters
    ----------
    angles_degrees : np.ndarray
        Array of shape ``(3,)`` containing rotation angles around the x, y,
        and z axes in degrees.

    Returns
    -------
    np.ndarray
        Rotation matrix of shape ``(3, 3)``.
    """

    ax, ay, az = np.deg2rad(np.asarray(angles_degrees, dtype=np.float64))

    rx = np.array(
        [
            [1.0, 0.0, 0.0],
            [0.0, np.cos(ax), -np.sin(ax)],
            [0.0, np.sin(ax), np.cos(ax)],
        ],
        dtype=np.float64,
    )
    ry = np.array(
        [
            [np.cos(ay), 0.0, np.sin(ay)],
            [0.0, 1.0, 0.0],
            [-np.sin(ay), 0.0, np.cos(ay)],
        ],
        dtype=np.float64,
    )
    rz = np.array(
        [
            [np.cos(az), -np.sin(az), 0.0],
            [np.sin(az), np.cos(az), 0.0],
            [0.0, 0.0, 1.0],
        ],
        dtype=np.float64,
    )
    return rz @ ry @ rx


def build_centered_rotated_affine(
    affine_in: np.ndarray,
    shape: tuple[int, ...],
    rotation_xyz: np.ndarray,
) -> np.ndarray:
    """Rotate a 3D image affine around the physical image center.

    Parameters
    ----------
    affine_in : np.ndarray
        Input homogeneous 4x4 voxel-to-world affine.
    shape : tuple[int, ...]
        Input image shape. The first three entries must be spatial.
    rotation_xyz : np.ndarray
        Rotation matrix of shape ``(3, 3)`` applied in world space.

    Returns
    -------
    np.ndarray
        Rotated homogeneous 4x4 affine whose world-space center is preserved.
    """

    affine_in = np.asarray(affine_in, dtype=np.float64)
    center_voxel = np.array(
        [
            (float(shape[0]) - 1.0) / 2.0,
            (float(shape[1]) - 1.0) / 2.0,
            (float(shape[2]) - 1.0) / 2.0,
            1.0,
        ],
        dtype=np.float64,
    )
    center_world = affine_in @ center_voxel

    rotation_4x4 = np.eye(4, dtype=np.float64)
    rotation_4x4[:3, :3] = rotation_xyz

    translate_to_center = np.eye(4, dtype=np.float64)
    translate_to_center[:3, 3] = -center_world[:3]

    translate_back = np.eye(4, dtype=np.float64)
    translate_back[:3, 3] = center_world[:3]

    return translate_back @ rotation_4x4 @ translate_to_center @ affine_in


def make_output_path(input_path: Path) -> Path:
    """Create the rotated output filename beside the input file.

    Parameters
    ----------
    input_path : Path
        Path to the original image file.

    Returns
    -------
    Path
        Output path with suffix ``_rotated.nii.gz``.
    """

    if input_path.name.endswith(".nii.gz"):
        stem = input_path.name[:-7]
        return input_path.with_name(f"{stem}_rotated.nii.gz")
    return input_path.with_name(f"{input_path.stem}_rotated.nii.gz")


rng = np.random.default_rng(RNG_SEED)
input_paths = [NOTEBOOK_DIR / name for name in INPUT_FILENAMES]
print("Notebook directory:", NOTEBOOK_DIR)
print("Random seed:", RNG_SEED)
print("Angle range (degrees):", ANGLE_RANGE_DEGREES)
for path in input_paths:
    print("  input:", path, "| exists=", path.exists())


Notebook directory: /homebase/code_sync/myMedIADIRLab/data_io/unittest_image
Random seed: 20260401
Angle range (degrees): (-22.0, 22.0)
  input: /homebase/code_sync/myMedIADIRLab/data_io/unittest_image/mni_icbm152_2009c_t1_1mm_img.nii.gz | exists= True
  input: /homebase/code_sync/myMedIADIRLab/data_io/unittest_image/HN_RRT_Jess_tp1_ct.nii.gz | exists= True
  input: /homebase/code_sync/myMedIADIRLab/data_io/unittest_image/OASIS_ORG_0001_orig.nii.gz | exists= True


In [2]:
missing_paths = [path for path in input_paths if not path.exists()]
if missing_paths:
    missing_str = "\n".join(str(path) for path in missing_paths)
    raise FileNotFoundError(f"Missing required input images:\n{missing_str}")

for input_path in input_paths:
    nii = nib.load(str(input_path))
    data = np.asarray(nii.dataobj)
    if data.ndim < 3:
        raise ValueError(f"Expected at least 3 spatial dimensions for {input_path.name}, got {data.ndim}")

    angles_deg = rng.uniform(
        low=ANGLE_RANGE_DEGREES[0],
        high=ANGLE_RANGE_DEGREES[1],
        size=3,
    )
    rotation_xyz = make_rotation_matrix_xyz(angles_deg)
    affine_rotated = build_centered_rotated_affine(
        affine_in=nii.affine,
        shape=tuple(int(v) for v in data.shape[:3]),
        rotation_xyz=rotation_xyz,
    )

    header_out = nii.header.copy()
    nii_rotated = nib.Nifti1Image(data, affine_rotated, header=header_out)
    nii_rotated.set_qform(affine_rotated, code=1)
    nii_rotated.set_sform(affine_rotated, code=1)

    output_path = make_output_path(input_path)
    nib.save(nii_rotated, str(output_path))

    print("=" * 100)
    print(f"Input:  {input_path.name}")
    print(f"Output: {output_path.name}")
    print("Angles (deg) [x, y, z]:", np.round(angles_deg, 3))
    print("Original axcodes:", nib.orientations.aff2axcodes(nii.affine))
    print("Rotated  axcodes:", nib.orientations.aff2axcodes(affine_rotated))
    print("Original affine:\n", np.array2string(nii.affine, precision=4, suppress_small=True))
    print("Rotated affine:\n", np.array2string(affine_rotated, precision=4, suppress_small=True))

print("Done. Add the generated *_rotated.nii.gz files to list_of_test_data.md when ready.")


Input:  mni_icbm152_2009c_t1_1mm_img.nii.gz
Output: mni_icbm152_2009c_t1_1mm_img_rotated.nii.gz
Angles (deg) [x, y, z]: [ 12.032  -0.585 -18.661]
Original axcodes: ('L', 'P', 'S')
Rotated  axcodes: ('L', 'P', 'S')
Original affine:
 [[ -1.   0.   0.  96.]
 [  0.  -1.   0.  96.]
 [  0.   0.   1. -78.]
 [  0.   0.   0.   1.]]
Rotated affine:
 [[ -0.9474  -0.3109  -0.0762 132.9162]
 [  0.32    -0.9273  -0.1943  75.8936]
 [ -0.0102  -0.2084   0.978  -51.6802]
 [  0.       0.       0.       1.    ]]
Input:  HN_RRT_Jess_tp1_ct.nii.gz
Output: HN_RRT_Jess_tp1_ct_rotated.nii.gz
Angles (deg) [x, y, z]: [  7.47  -12.099   4.764]
Original axcodes: ('L', 'P', 'S')
Rotated  axcodes: ('L', 'P', 'S')
Original affine:
 [[  -0.9766    0.        0.      249.5117]
 [   0.       -0.9766    0.      434.0117]
 [   0.        0.        3.     -689.5   ]
 [   0.        0.        0.        1.    ]]
Rotated affine:
 [[  -0.9516    0.1069   -0.5889  253.4965]
 [  -0.0793   -0.9627   -0.4404  478.9248]
 [  -0.2047  